# Q4 Training EN

This notebook runs the full Question 4 pipeline for **English** only.

It does all of the following in one place:
- trains the English NNLM
- saves a resumable checkpoint every 5 epochs
- resumes automatically from `checkpoint_nnlm_en.pt` after a crash
- computes the English Q4 tables, plots, analogy output, and word2vec comparison
- packages everything into `q4_en_artifacts.zip`

You do **not** need to run the Question 3 notebook first inside this session.
The shared Q3/Q4 setup is recreated internally by `q4_parallel_runtime.py`
using the same seed, tokenizer, corpus-cache names, and preprocessing rules.


## Install Packages


In [ ]:
import subprocess
import sys

packages = [
    "datasets",
    "indic-nlp-library",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "nltk",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


## Bootstrap Shared Runtime

This cell recreates `q4_parallel_runtime.py` automatically inside Kaggle,
so the notebook works even if you uploaded only the `.ipynb` file.


In [ ]:
from pathlib import Path

runtime_path = Path("q4_parallel_runtime.py")
runtime_path.write_text('from __future__ import annotations\n\nimport copy\nimport json\nimport math\nimport multiprocessing as mp\nimport os\nimport pickle\nimport random\nimport re\nimport shutil\nimport zipfile\nfrom collections import Counter\nfrom getpass import getpass\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport nltk\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom datasets import load_dataset\n\n\nnltk.download("punkt", quiet=True)\ntry:\n    nltk.download("punkt_tab", quiet=True)\nexcept Exception:\n    pass\n\n\nLANGS = ("en", "hi")\nSPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]\nSPECIAL_SET = set(SPECIALS)\n\nSHARED_SEED = 42\nSHARED_N_PAIRS = None\nMIN_TOKENS = 3\nSHARED_MIN_COUNT = 2\n\nEN_CACHE = "shared_english_corpus.csv"\nHI_CACHE = "shared_hindi_corpus.csv"\n\nEMBED_DIM = 100\nDEFAULT_Q4_N = 4\nDEFAULT_Q4_HIDDEN = 100\nDEFAULT_Q4_EPOCHS = 30\nDEFAULT_Q4_BATCH_SIZE = 512\nDEFAULT_Q4_LR = 1e-3\nDEFAULT_Q4_WEIGHT_DECAY = 1e-4\nDEFAULT_Q4_PATIENCE = 3\nDEFAULT_DIRECT = False\n\nSTAGE1_BUNDLE = "q4_stage1_bundle.zip"\nSTAGE2_BUNDLE = "q4_stage2_bundle.zip"\nFINAL_BUNDLE = "question_4_bundle.zip"\nCHECKPOINT_EVERY = 5\n\n\ndef build_default_config():\n    return {\n        "n": DEFAULT_Q4_N,\n        "m": EMBED_DIM,\n        "h": DEFAULT_Q4_HIDDEN,\n        "epochs": DEFAULT_Q4_EPOCHS,\n        "batch_size": DEFAULT_Q4_BATCH_SIZE,\n        "lr": DEFAULT_Q4_LR,\n        "weight_decay": DEFAULT_Q4_WEIGHT_DECAY,\n        "patience": DEFAULT_Q4_PATIENCE,\n        "min_freq": SHARED_MIN_COUNT,\n        "direct": DEFAULT_DIRECT,\n        "max_pairs": None,\n        "preferred_q3_method": {"en": None, "hi": None},\n        "hf_token": os.environ.get("HF_TOKEN", "").strip(),\n    }\n\n\ndef language_label(lang: str) -> str:\n    return "English" if lang == "en" else "Hindi"\n\n\ndef set_shared_seed(seed: int = SHARED_SEED):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    return seed\n\n\ndef shared_tokenize(text):\n    return nltk.word_tokenize(str(text).lower())\n\n\ndef get_hf_token(config: dict | None = None) -> str:\n    env_token = os.environ.get("HF_TOKEN", "").strip()\n    if env_token:\n        return env_token\n\n    if config and config.get("hf_token"):\n        token = str(config["hf_token"]).strip()\n        if token:\n            os.environ["HF_TOKEN"] = token\n            return token\n\n    token = getpass("Enter Hugging Face token: ").strip()\n    if not token:\n        raise ValueError("A Hugging Face token is required to download the assignment dataset.")\n    os.environ["HF_TOKEN"] = token\n    return token\n\n\ndef prime_hf_token(config: dict | None = None):\n    if Path(EN_CACHE).exists() and Path(HI_CACHE).exists():\n        return\n    token = get_hf_token(config)\n    os.environ["HF_TOKEN"] = token\n\n\ndef load_shared_corpus(\n    n_pairs=SHARED_N_PAIRS,\n    seed: int = SHARED_SEED,\n    en_cache: str = EN_CACHE,\n    hi_cache: str = HI_CACHE,\n    config: dict | None = None,\n):\n    if os.path.exists(en_cache) and os.path.exists(hi_cache):\n        en_df = pd.read_csv(en_cache, keep_default_na=False)\n        hi_df = pd.read_csv(hi_cache, keep_default_na=False)\n        print(f"Loaded English corpus from {en_cache}")\n        print(f"Loaded Hindi corpus from {hi_cache}")\n        return en_df, hi_df\n\n    ds = load_dataset(\n        "Exploration-Lab/CS781",\n        data_files={"train": "Assignment-1/train.parquet"},\n        split="train",\n        token=get_hf_token(config),\n    )\n\n    translations = ds["translation"]\n    en_df = pd.DataFrame({"text": [row["en"] for row in translations]})\n    hi_df = pd.DataFrame({"text": [row["hi"] for row in translations]})\n\n    en_df = en_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])\n    hi_df = hi_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])\n\n    en_df = en_df[en_df["text"].apply(lambda x: len(shared_tokenize(x)) >= MIN_TOKENS)]\n    hi_df = hi_df[hi_df["text"].apply(lambda x: len(shared_tokenize(x)) >= MIN_TOKENS)]\n\n    en_order = np.random.RandomState(seed).permutation(len(en_df))\n    hi_order = np.random.RandomState(seed).permutation(len(hi_df))\n    en_df = en_df.iloc[en_order].reset_index(drop=True)\n    hi_df = hi_df.iloc[hi_order].reset_index(drop=True)\n\n    if n_pairs is not None:\n        en_df = en_df.iloc[:n_pairs].reset_index(drop=True)\n        hi_df = hi_df.iloc[:n_pairs].reset_index(drop=True)\n\n    en_df.to_csv(en_cache, index=False)\n    hi_df.to_csv(hi_cache, index=False)\n    print(f"Wrote {len(en_df):,} English sentences to {en_cache}")\n    print(f"Wrote {len(hi_df):,} Hindi sentences to {hi_cache}")\n    return en_df, hi_df\n\n\ndef load_language_texts(lang: str, config: dict) -> list[str]:\n    set_shared_seed()\n    en_df, hi_df = load_shared_corpus(config=config)\n    texts = en_df["text"].tolist() if lang == "en" else hi_df["text"].tolist()\n    max_pairs = config.get("max_pairs")\n    return texts[:max_pairs] if max_pairs is not None else texts\n\n\ndef tokenize(text):\n    return shared_tokenize(text)\n\n\ndef split_sentences(sentences, ratios=(0.8, 0.1, 0.1)):\n    total = len(sentences)\n    train_end = int(total * ratios[0])\n    valid_end = train_end + int(total * ratios[1])\n    train = sentences[:train_end]\n    valid = sentences[train_end:valid_end]\n    test = sentences[valid_end:]\n    return train, valid, test\n\n\ndef build_vocabulary(sentences, min_freq=SHARED_MIN_COUNT):\n    counter = Counter(token for sent in sentences for token in sent)\n    kept = [tok for tok, freq in counter.items() if freq >= min_freq and tok not in SPECIAL_SET]\n    kept.sort(key=lambda tok: (-counter[tok], tok))\n    itos = SPECIALS + kept\n    stoi = {tok: idx for idx, tok in enumerate(itos)}\n    return stoi, itos, counter\n\n\ndef unk_rate(sentences, stoi):\n    total = sum(len(sent) for sent in sentences)\n    if total == 0:\n        return 0.0\n    unknown = sum(1 for sent in sentences for token in sent if token not in stoi)\n    return unknown / total\n\n\ndef encode_sentence(sentence, stoi):\n    unk_id = stoi["<unk>"]\n    return [stoi.get(token, unk_id) for token in sentence]\n\n\ndef map_sentence_to_vocab(sentence, stoi):\n    return [token if token in stoi else "<unk>" for token in sentence]\n\n\ndef make_windows(sentences, stoi, n):\n    bos = [stoi["<s>"]] * (n - 1)\n    eos = stoi["</s>"]\n    xs = []\n    ys = []\n    for sentence in sentences:\n        ids = bos + encode_sentence(sentence, stoi) + [eos]\n        for end in range(n - 1, len(ids)):\n            gram = ids[end - (n - 1): end + 1]\n            xs.append(gram[:-1])\n            ys.append(gram[-1])\n\n    if not xs:\n        return (\n            np.empty((0, n - 1), dtype=np.int32),\n            np.empty((0,), dtype=np.int32),\n        )\n\n    return np.asarray(xs, dtype=np.int32), np.asarray(ys, dtype=np.int32)\n\n\ndef padded_token_sentences(sentences, stoi, n):\n    bos = ["<s>"] * (n - 1)\n    eos = ["</s>"]\n    return [bos + map_sentence_to_vocab(sent, stoi) + eos for sent in sentences]\n\n\ndef prepare_language_resources(lang: str, config: dict):\n    texts = load_language_texts(lang, config)\n    tokenized = [tokenize(text) for text in texts if str(text).strip()]\n    train_sentences, valid_sentences, test_sentences = split_sentences(tokenized)\n    stoi, itos, counter = build_vocabulary(train_sentences, min_freq=config["min_freq"])\n\n    n = config["n"]\n    Xtr, Ytr = make_windows(train_sentences, stoi, n=n)\n    Xva, Yva = make_windows(valid_sentences, stoi, n=n)\n    Xte, Yte = make_windows(test_sentences, stoi, n=n)\n\n    return {\n        "texts": texts,\n        "tokenized": tokenized,\n        "train_sentences": train_sentences,\n        "valid_sentences": valid_sentences,\n        "test_sentences": test_sentences,\n        "stoi": stoi,\n        "itos": itos,\n        "counter": counter,\n        "Xtr": Xtr,\n        "Ytr": Ytr,\n        "Xva": Xva,\n        "Yva": Yva,\n        "Xte": Xte,\n        "Yte": Yte,\n        "unk_rates": {\n            "train": unk_rate(train_sentences, stoi),\n            "valid": unk_rate(valid_sentences, stoi),\n            "test": unk_rate(test_sentences, stoi),\n        },\n        "split_sizes": {\n            "sentences_train": len(train_sentences),\n            "sentences_valid": len(valid_sentences),\n            "sentences_test": len(test_sentences),\n        },\n        "window_counts": {\n            "train": int(len(Ytr)),\n            "valid": int(len(Yva)),\n            "test": int(len(Yte)),\n        },\n    }\n\n\ndef resolve_device(device_id: int | None):\n    if device_id is not None and device_id >= 0 and torch.cuda.is_available():\n        usable = min(device_id, torch.cuda.device_count() - 1)\n        torch.cuda.set_device(usable)\n        device = torch.device(f"cuda:{usable}")\n        print(f"Using {device} -> {torch.cuda.get_device_name(usable)}")\n        return device\n\n    if torch.cuda.is_available():\n        device = torch.device("cuda:0")\n        torch.cuda.set_device(0)\n        print(f"Using {device} -> {torch.cuda.get_device_name(0)}")\n        return device\n\n    print("CUDA not available, using CPU.")\n    return torch.device("cpu")\n\n\nclass NNLM(nn.Module):\n    def __init__(self, V, m=100, h=100, n=4, direct=False):\n        super().__init__()\n        self.V = V\n        self.m = m\n        self.h = h\n        self.n = n\n        self.direct = direct\n        self.context_dim = (n - 1) * m\n\n        self.C = nn.Embedding(V, m)\n        self.H = nn.Linear(self.context_dim, h, bias=False)\n        self.d = nn.Parameter(torch.zeros(h))\n        self.U = nn.Linear(h, V, bias=False)\n        self.W = nn.Linear(self.context_dim, V, bias=False) if direct else None\n        self.b = nn.Parameter(torch.zeros(V))\n\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.uniform_(self.C.weight, -0.1, 0.1)\n        nn.init.xavier_uniform_(self.H.weight)\n        nn.init.xavier_uniform_(self.U.weight)\n        if self.W is not None:\n            nn.init.xavier_uniform_(self.W.weight)\n        nn.init.zeros_(self.d)\n        nn.init.zeros_(self.b)\n\n    def forward(self, context):\n        x = self.C(context).reshape(context.size(0), -1)\n        hidden_pre = self.H(x) + self.d\n        hidden = torch.tanh(hidden_pre)\n        hidden_term = self.U(hidden)\n        direct_term = self.W(x) if self.W is not None else 0.0\n        return self.b + direct_term + hidden_term\n\n    def embeddings(self):\n        return self.C.weight.detach().cpu().numpy().astype(np.float32).copy()\n\n\ndef batches(X, Y, batch_size, device, shuffle=True):\n    size = len(Y)\n    if size == 0:\n        return\n    indices = np.random.permutation(size) if shuffle else np.arange(size)\n    for start in range(0, size, batch_size):\n        sl = indices[start:start + batch_size]\n        xb = torch.from_numpy(X[sl]).long().to(device, non_blocking=True)\n        yb = torch.from_numpy(Y[sl]).long().to(device, non_blocking=True)\n        yield xb, yb\n\n\n@torch.no_grad()\ndef mean_nll(model, X, Y, device, batch_size=2048):\n    model.eval()\n    total_nll = 0.0\n    total_items = 0\n    for xb, yb in batches(X, Y, batch_size=batch_size, device=device, shuffle=False):\n        logits = model(xb)\n        loss_sum = F.cross_entropy(logits, yb, reduction="sum")\n        total_nll += float(loss_sum.detach().cpu())\n        total_items += int(yb.size(0))\n    if total_items == 0:\n        return float("inf")\n    return total_nll / total_items\n\n\n@torch.no_grad()\ndef perplexity(model, X, Y, device, batch_size=2048):\n    avg_nll = mean_nll(model, X, Y, device=device, batch_size=batch_size)\n    return float(np.exp(min(avg_nll, 50.0)))\n\n\ndef train_model(model, Xtr, Ytr, Xva, Yva, device, config, verbose=True, checkpoint_lang=None, stoi=None, itos=None):\n    decay_params = [model.C.weight, model.H.weight, model.U.weight]\n    if model.W is not None:\n        decay_params.append(model.W.weight)\n\n    no_decay_params = [model.d, model.b]\n    optimizer = optim.Adam(\n        [\n            {"params": decay_params, "weight_decay": config["weight_decay"]},\n            {"params": no_decay_params, "weight_decay": 0.0},\n        ],\n        lr=config["lr"],\n    )\n\n    history = {\n        "train_loss": [],\n        "train_ppl": [],\n        "valid_loss": [],\n        "valid_ppl": [],\n        "best_epoch": 0,\n        "best_valid_ppl": float("inf"),\n        "converged": False,\n        "stopped_epoch": 0,\n    }\n\n    start_epoch = 0\n    best_state = copy.deepcopy(model.state_dict())\n    stale_epochs = 0\n    if checkpoint_lang is not None and stoi is not None and itos is not None:\n        start_epoch, resume_state = load_resume_checkpoint(\n            checkpoint_lang, model, optimizer, device, stoi, itos, config\n        )\n        if resume_state is not None:\n            history = resume_state["history"]\n            best_state = resume_state["best_state_dict"]\n            history["best_valid_ppl"] = resume_state.get("best_valid_ppl", history["best_valid_ppl"])\n            stale_epochs = int(resume_state.get("stale_epochs", 0))\n            print(\n                f"[checkpoint] resumed {checkpoint_lang} from epoch {start_epoch} "\n                f"(best valid ppl {history[\'best_valid_ppl\']:.2f})"\n            )\n\n    for epoch in range(start_epoch + 1, config["epochs"] + 1):\n        model.train()\n        total_nll = 0.0\n        total_items = 0\n        for xb, yb in batches(Xtr, Ytr, batch_size=config["batch_size"], device=device, shuffle=True):\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(xb)\n            loss = F.cross_entropy(logits, yb, reduction="mean")\n            loss.backward()\n            optimizer.step()\n\n            batch_items = int(yb.size(0))\n            total_items += batch_items\n            total_nll += float(loss.detach().cpu()) * batch_items\n\n        train_loss = total_nll / max(total_items, 1)\n        valid_loss = mean_nll(model, Xva, Yva, device=device, batch_size=config["batch_size"])\n        train_ppl = float(np.exp(min(train_loss, 50.0)))\n        valid_ppl = float(np.exp(min(valid_loss, 50.0)))\n\n        history["train_loss"].append(train_loss)\n        history["train_ppl"].append(train_ppl)\n        history["valid_loss"].append(valid_loss)\n        history["valid_ppl"].append(valid_ppl)\n\n        if verbose:\n            print(\n                f"Epoch {epoch:02d} | train loss={train_loss:.4f} | train ppl={train_ppl:.2f} | "\n                f"valid loss={valid_loss:.4f} | valid ppl={valid_ppl:.2f}"\n            )\n\n        if valid_ppl + 1e-8 < history["best_valid_ppl"]:\n            history["best_valid_ppl"] = valid_ppl\n            history["best_epoch"] = epoch\n            best_state = copy.deepcopy(model.state_dict())\n            stale_epochs = 0\n        else:\n            stale_epochs += 1\n            if stale_epochs >= config["patience"]:\n                history["converged"] = True\n                history["stopped_epoch"] = epoch\n                if checkpoint_lang is not None and stoi is not None and itos is not None:\n                    save_resume_checkpoint(\n                        checkpoint_lang,\n                        epoch,\n                        model,\n                        optimizer,\n                        history,\n                        best_state,\n                        history["best_valid_ppl"],\n                        stale_epochs,\n                        config,\n                        stoi,\n                        itos,\n                    )\n                break\n\n        if checkpoint_lang is not None and stoi is not None and itos is not None:\n            if epoch % CHECKPOINT_EVERY == 0 or epoch == config["epochs"]:\n                save_resume_checkpoint(\n                    checkpoint_lang,\n                    epoch,\n                    model,\n                    optimizer,\n                    history,\n                    best_state,\n                    history["best_valid_ppl"],\n                    stale_epochs,\n                    config,\n                    stoi,\n                    itos,\n                )\n\n    if not history["stopped_epoch"]:\n        history["stopped_epoch"] = len(history["train_ppl"])\n\n    model.load_state_dict(best_state)\n    return history\n\n\ndef save_figure(fig, path):\n    path = Path(path)\n    fig.tight_layout()\n    fig.savefig(path, dpi=180, bbox_inches="tight")\n    plt.close(fig)\n    return str(path)\n\n\ndef plot_convergence(history, lang, save_path):\n    epochs = np.arange(1, len(history["train_ppl"]) + 1)\n    best_epoch = history["best_epoch"]\n    status = "converged" if history["converged"] else "max-epochs reached"\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    axes[0].plot(epochs, history["train_loss"], marker="o", label="train loss")\n    axes[0].plot(epochs, history["valid_loss"], marker="o", label="valid loss")\n    axes[0].axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.7)\n    axes[0].set_title(f"{language_label(lang)} | Loss")\n    axes[0].set_xlabel("Epoch")\n    axes[0].set_ylabel("Cross-Entropy")\n    axes[0].legend()\n\n    axes[1].plot(epochs, history["train_ppl"], marker="o", label="train ppl")\n    axes[1].plot(epochs, history["valid_ppl"], marker="o", label="valid ppl")\n    axes[1].axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.7)\n    axes[1].set_title(f"{language_label(lang)} | Perplexity ({status})")\n    axes[1].set_xlabel("Epoch")\n    axes[1].set_ylabel("Perplexity")\n    axes[1].legend()\n    return save_figure(fig, save_path)\n\n\ndef count_parameters(V, m, h, n, direct=False):\n    context_dim = (n - 1) * m\n    counts = {\n        "C": V * m,\n        "H": h * context_dim,\n        "d": h,\n        "U": V * h,\n        "W": V * context_dim if direct else 0,\n        "b": V,\n    }\n    counts["TOTAL"] = sum(counts.values())\n    counts["QUESTION_FORMULA_IF_DIRECT"] = V * (1 + n * m + h) + h * (1 + context_dim)\n    counts["FORMULA_WITH_CURRENT_SETTING"] = (\n        V * (1 + m + h + (context_dim if direct else 0)) + h * (1 + context_dim)\n    )\n    return counts\n\n\ndef generate_ngrams(tokens, n):\n    if len(tokens) < n:\n        return []\n    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]\n\n\nclass Q2_NGramLM:\n    def __init__(self, n, V):\n        self.n = n\n        self.V = V\n        self.ngram_counts = {order: Counter() for order in range(1, n + 1)}\n        self.context_counts = {order: Counter() for order in range(1, n)}\n        self.total_unigrams = 0\n\n    def fit(self, padded_sentences):\n        for sent in padded_sentences:\n            for full_ngram in generate_ngrams(sent, self.n):\n                for order in range(1, self.n + 1):\n                    gram = tuple(full_ngram[-order:])\n                    self.ngram_counts[order][gram] += 1\n\n        self.total_unigrams = sum(self.ngram_counts[1].values())\n        for order in range(2, self.n + 1):\n            for gram, count in self.ngram_counts[order].items():\n                self.context_counts[order - 1][gram[:-1]] += count\n        return self\n\n    def prob(self, ctx, w, mode="interp", alpha=1.0):\n        ctx = list(ctx)[-(self.n - 1):]\n        w = str(w)\n        if mode == "laplace":\n            top_ctx = tuple(ctx[-(self.n - 1):])\n            gram = top_ctx + (w,)\n            numerator = self.ngram_counts[self.n].get(gram, 0) + alpha\n            denominator = self.context_counts[self.n - 1].get(top_ctx, 0) + alpha * self.V\n            return float(numerator / denominator)\n\n        pieces = [1.0 / self.V]\n        unigram_prob = (\n            self.ngram_counts[1].get((w,), 0) / self.total_unigrams\n            if self.total_unigrams\n            else 1.0 / self.V\n        )\n        pieces.append(unigram_prob)\n        for order in range(2, self.n + 1):\n            local_ctx = tuple(ctx[-(order - 1):])\n            gram = local_ctx + (w,)\n            denom = self.context_counts[order - 1].get(local_ctx, 0)\n            prob = self.ngram_counts[order].get(gram, 0) / denom if denom else 0.0\n            pieces.append(prob)\n        return float(sum(pieces) / len(pieces))\n\n    def token_probs(self, padded_sentences, mode="interp"):\n        probs = []\n        for sent in padded_sentences:\n            for full_ngram in generate_ngrams(sent, self.n):\n                ctx = full_ngram[:-1]\n                w = full_ngram[-1]\n                probs.append(self.prob(ctx, w, mode=mode))\n        return np.asarray(probs, dtype=np.float64)\n\n    def perplexity(self, padded_sentences, mode="interp"):\n        return ppl_from_probs(self.token_probs(padded_sentences, mode=mode))\n\n    def distinct_ngrams_total(self):\n        return sum(len(counter) for counter in self.ngram_counts.values())\n\n\ndef ppl_from_probs(p):\n    p = np.asarray(p, dtype=np.float64)\n    if p.size == 0:\n        return float("inf")\n    p = np.clip(p, 1e-12, 1.0)\n    return float(np.exp(-np.mean(np.log(p))))\n\n\n@torch.no_grad()\ndef nnlm_token_probs(model, X, Y, device, batch_size=2048):\n    model.eval()\n    parts = []\n    for xb, yb in batches(X, Y, batch_size=batch_size, device=device, shuffle=False):\n        logits = model(xb)\n        log_probs = F.log_softmax(logits, dim=1)\n        chosen = log_probs.gather(1, yb.unsqueeze(1)).squeeze(1).exp()\n        parts.append(chosen.detach().cpu().numpy())\n    if not parts:\n        return np.asarray([], dtype=np.float64)\n    return np.concatenate(parts).astype(np.float64)\n\n\ndef mixture_report_from_probs(nnlm_probs, ngram_probs, lam=0.5):\n    nnlm_probs = np.asarray(nnlm_probs, dtype=np.float64)\n    ngram_probs = np.asarray(ngram_probs, dtype=np.float64)\n    assert len(nnlm_probs) == len(ngram_probs), "Probability arrays must align token-by-token."\n\n    grid_rows = []\n    best_lambda = None\n    best_ppl = float("inf")\n    for candidate in np.round(np.arange(0.0, 1.0001, 0.05), 2):\n        mix = candidate * nnlm_probs + (1.0 - candidate) * ngram_probs\n        ppl = ppl_from_probs(mix)\n        grid_rows.append({"lambda": float(candidate), "mixture_ppl": float(ppl)})\n        if ppl < best_ppl:\n            best_lambda = float(candidate)\n            best_ppl = float(ppl)\n\n    lam_mix = lam * nnlm_probs + (1.0 - lam) * ngram_probs\n    return {\n        "nnlm_ppl": ppl_from_probs(nnlm_probs),\n        "interp_ngram_ppl": ppl_from_probs(ngram_probs),\n        "mixture_ppl_at_lambda": ppl_from_probs(lam_mix),\n        "best_lambda": best_lambda,\n        "best_mixture_ppl": best_ppl,\n        "grid": pd.DataFrame(grid_rows),\n    }\n\n\ndef sparsity_table(Xtr, Ytr, Xte, Yte):\n    rows = []\n    max_order = Xtr.shape[1] + 1\n    test_tokens = int(len(Yte))\n    for order in range(2, max_order + 1):\n        ctx_len = order - 1\n        train_contexts = {tuple(row[:ctx_len].tolist()) for row in Xtr}\n        train_ngrams = {\n            tuple(list(row[:ctx_len]) + [int(y)])\n            for row, y in zip(Xtr, Ytr)\n        }\n        unseen_context = 0\n        unseen_ngram = 0\n        for row, y in zip(Xte, Yte):\n            ctx = tuple(row[:ctx_len].tolist())\n            gram = ctx + (int(y),)\n            if ctx not in train_contexts:\n                unseen_context += 1\n            if gram not in train_ngrams:\n                unseen_ngram += 1\n        rows.append(\n            {\n                "n": order,\n                "train_contexts_seen": len(train_contexts),\n                "train_ngrams_seen": len(train_ngrams),\n                "test_tokens": test_tokens,\n                "pct_unseen_context": 100.0 * unseen_context / max(test_tokens, 1),\n                "pct_unseen_ngram": 100.0 * unseen_ngram / max(test_tokens, 1),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef normalize_rows(matrix):\n    matrix = np.asarray(matrix, dtype=np.float32)\n    norms = np.linalg.norm(matrix, axis=1, keepdims=True)\n    norms = np.clip(norms, 1e-12, None)\n    return matrix / norms\n\n\ndef cosine(u, v):\n    u = np.asarray(u, dtype=np.float32)\n    v = np.asarray(v, dtype=np.float32)\n    denom = np.linalg.norm(u) * np.linalg.norm(v)\n    if denom <= 1e-12:\n        return 0.0\n    return float(np.dot(u, v) / denom)\n\n\ndef is_content_token(token):\n    if token in SPECIAL_SET:\n        return False\n    if token == "\\u0964":\n        return False\n    return re.search(r"\\w", token, flags=re.UNICODE) is not None\n\n\ndef pick_words(itos, start=50, stop=1500, k=18):\n    candidates = [tok for tok in itos if is_content_token(tok)]\n    if not candidates:\n        return []\n    upper = min(stop, len(candidates))\n    lower = min(start, upper - 1) if upper > 0 else 0\n    pool = candidates[lower:upper] if upper > lower else candidates\n    if len(pool) <= k:\n        return pool\n    indices = np.linspace(0, len(pool) - 1, num=k, dtype=int)\n    return [pool[idx] for idx in indices]\n\n\ndef plot_pca(E, stoi, words, title, save_path):\n    labels = [word for word in words if word in stoi]\n    if len(labels) < 2:\n        print(f"Skipping PCA for {title}: need at least 2 in-vocabulary words.")\n        return None\n    vectors = np.stack([E[stoi[word]] for word in labels], axis=0)\n    centered = vectors - vectors.mean(axis=0, keepdims=True)\n    _, _, vt = np.linalg.svd(centered, full_matrices=False)\n    coords = centered @ vt[:2].T\n\n    fig, ax = plt.subplots(figsize=(8.5, 6.5))\n    ax.scatter(coords[:, 0], coords[:, 1], s=28, alpha=0.8, color="tab:blue")\n    for idx, word in enumerate(labels):\n        ax.text(coords[idx, 0], coords[idx, 1], word, fontsize=8)\n    ax.set_title(title)\n    ax.set_xlabel("PC1")\n    ax.set_ylabel("PC2")\n    return save_figure(fig, save_path)\n\n\ndef load_word2vec(path):\n    with Path(path).open("rb") as f:\n        data = pickle.load(f)\n    raw_itos = data.get("itos", data.get("vocab"))\n    if isinstance(raw_itos, dict):\n        itos = [raw_itos[idx] for idx in range(len(raw_itos))]\n    else:\n        itos = list(raw_itos)\n    return {\n        "path": str(path),\n        "embeddings": np.asarray(data["embeddings"], dtype=np.float32),\n        "stoi": data["stoi"],\n        "itos": itos,\n    }\n\n\ndef nearest_neighbours(word, E, stoi, itos, k=10):\n    word = str(word).lower()\n    if word not in stoi:\n        return []\n    norm = normalize_rows(E)\n    sims = norm @ norm[stoi[word]]\n    sims[stoi[word]] = -np.inf\n    for special in SPECIALS:\n        if special in stoi:\n            sims[stoi[special]] = -np.inf\n    ordered = np.argsort(-sims)\n    out = []\n    for idx in ordered:\n        token = itos[int(idx)]\n        if not is_content_token(token):\n            continue\n        out.append((token, float(sims[int(idx)])))\n        if len(out) == k:\n            break\n    return out\n\n\ndef same_word_similarity(E_a, stoi_a, E_b, stoi_b, words):\n    scores = []\n    for word in words:\n        if word in stoi_a and word in stoi_b:\n            scores.append(cosine(E_a[stoi_a[word]], E_b[stoi_b[word]]))\n    return float(np.mean(scores)) if scores else None\n\n\ndef load_analogy_set(lang, config):\n    split = load_dataset(\n        "Exploration-Lab/CS781",\n        data_files={lang: f"Assignment-1/analogy_test_{lang}.parquet"},\n        split=lang,\n        token=get_hf_token(config),\n    )\n    df = split.to_pandas()\n    if {"word1", "word2", "word3"}.issubset(df.columns):\n        return df[["word1", "word2", "word3"]].copy()\n    cols = list(df.columns[:3])\n    renamed = df[cols].copy()\n    renamed.columns = ["word1", "word2", "word3"]\n    return renamed\n\n\ndef embedding_analogy_predict(E, E_norm, stoi, itos, w1, w2, w3):\n    words = [str(w1).lower(), str(w2).lower(), str(w3).lower()]\n    if any(word not in stoi for word in words):\n        return "OOV"\n    idx1, idx2, idx3 = (stoi[word] for word in words)\n    query = E[idx1] - E[idx2] + E[idx3]\n    qnorm = np.linalg.norm(query)\n    if qnorm <= 1e-12:\n        return "OOV"\n    sims = E_norm @ (query / qnorm)\n    for idx in (idx1, idx2, idx3):\n        sims[idx] = -np.inf\n    for special in SPECIALS:\n        if special in stoi:\n            sims[stoi[special]] = -np.inf\n    best = int(np.argmax(sims))\n    return itos[best] if np.isfinite(sims[best]) else "OOV"\n\n\ndef extract_bundle_if_present(bundle_name):\n    bundle = Path(bundle_name)\n    if bundle.exists():\n        with zipfile.ZipFile(bundle, "r") as zf:\n            zf.extractall(".")\n        print(f"Extracted {bundle_name}")\n\n\ndef stage_kaggle_input_files(required_names):\n    input_root = Path("/kaggle/input")\n    if not input_root.exists():\n        return\n\n    for name in required_names:\n        local_path = Path(name)\n        if local_path.exists():\n            continue\n\n        matches = list(input_root.rglob(name))\n        if not matches:\n            continue\n\n        source = matches[0]\n        shutil.copy2(source, local_path)\n        print(f"Copied input file {source} -> {local_path}")\n\n\ndef train_files(lang):\n    return {\n        "model": Path(f"nnlm_{lang}.pt"),\n        "checkpoint": Path(f"checkpoint_nnlm_{lang}.pt"),\n        "state": Path(f"q4_train_state_{lang}.pkl"),\n        "summary": Path(f"q4_train_summary_{lang}.json"),\n        "plot": Path(f"q4_convergence_{lang}.png"),\n        "bundle": Path(f"q4_{lang}_artifacts.zip"),\n    }\n\n\ndef metrics_files(lang):\n    return {\n        "perplexity": Path(f"q4_perplexity_table_{lang}.csv"),\n        "parameter": Path(f"q4_parameter_table_{lang}.csv"),\n        "sparsity": Path(f"q4_sparsity_table_{lang}.csv"),\n        "mixture": Path(f"q4_mixture_grid_{lang}.csv"),\n        "summary": Path(f"q4_metrics_summary_{lang}.json"),\n    }\n\n\ndef analysis_files(lang):\n    return {\n        "analogy": Path(f"analogy_results_nnlm_{lang}.csv"),\n        "pca": Path(f"q4_embedding_pca_{lang}.png"),\n        "word2vec_compare": Path(f"q4_word2vec_compare_{lang}.csv"),\n        "word2vec_proxy": Path(f"q4_word2vec_proxy_{lang}.csv"),\n        "embed_compare_plot": Path(f"q4_embedding_compare_{lang}.png"),\n        "written_answers": Path(f"Q4_WRITTEN_ANSWERS_{lang}.md"),\n        "summary": Path(f"q4_analysis_summary_{lang}.json"),\n    }\n\n\ndef json_ready(value):\n    if isinstance(value, (np.floating, np.integer)):\n        return value.item()\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, pd.DataFrame):\n        return value.to_dict(orient="records")\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, dict):\n        return {str(k): json_ready(v) for k, v in value.items()}\n    if isinstance(value, list):\n        return [json_ready(v) for v in value]\n    return value\n\n\ndef save_json(path, data):\n    Path(path).write_text(json.dumps(json_ready(data), indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef save_model_checkpoint(lang, model, config, stoi, itos):\n    files = train_files(lang)\n    cpu_state = {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()}\n    torch.save(\n        {\n            "state_dict": cpu_state,\n            "config": {\n                "V": len(itos),\n                "m": config["m"],\n                "h": config["h"],\n                "n": config["n"],\n                "direct": config["direct"],\n            },\n            "stoi": stoi,\n            "itos": itos,\n        },\n        files["model"],\n    )\n\n\ndef save_resume_checkpoint(\n    lang,\n    epoch,\n    model,\n    optimizer,\n    history,\n    best_state,\n    best_valid_ppl,\n    stale_epochs,\n    config,\n    stoi,\n    itos,\n):\n    checkpoint_path = train_files(lang)["checkpoint"]\n    model_state = {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()}\n    best_state_cpu = {name: tensor.detach().cpu() for name, tensor in best_state.items()}\n    optimizer_state = optimizer.state_dict()\n    for state in optimizer_state.get("state", {}).values():\n        for key, value in list(state.items()):\n            if torch.is_tensor(value):\n                state[key] = value.detach().cpu()\n\n    torch.save(\n        {\n            "epoch": epoch,\n            "model_state_dict": model_state,\n            "optimizer_state_dict": optimizer_state,\n            "history": history,\n            "best_state_dict": best_state_cpu,\n            "best_valid_ppl": best_valid_ppl,\n            "stale_epochs": stale_epochs,\n            "config": {\n                "n": config["n"],\n                "m": config["m"],\n                "h": config["h"],\n                "direct": config["direct"],\n                "min_freq": config["min_freq"],\n                "epochs": config["epochs"],\n                "batch_size": config["batch_size"],\n                "lr": config["lr"],\n                "weight_decay": config["weight_decay"],\n                "patience": config["patience"],\n            },\n            "stoi": stoi,\n            "itos": itos,\n        },\n        checkpoint_path,\n    )\n    print(f"[checkpoint] saved -> {checkpoint_path.name} (epoch {epoch})")\n\n\ndef load_resume_checkpoint(lang, model, optimizer, device, stoi, itos, config):\n    checkpoint_path = train_files(lang)["checkpoint"]\n    if not checkpoint_path.exists():\n        return 0, None\n\n    ckpt = torch.load(checkpoint_path, map_location=device)\n    if ckpt.get("stoi") != stoi or ckpt.get("itos") != itos:\n        raise ValueError(f"[checkpoint] vocabulary mismatch for {lang}.")\n\n    saved_cfg = ckpt.get("config", {})\n    for key in ("n", "m", "h", "direct", "min_freq"):\n        if saved_cfg.get(key) != config.get(key):\n            raise ValueError(f"[checkpoint] config mismatch for {lang} on \'{key}\'.")\n\n    model.load_state_dict(ckpt["model_state_dict"])\n    optimizer.load_state_dict(ckpt["optimizer_state_dict"])\n    return int(ckpt.get("epoch", 0)), ckpt\n\n\ndef load_model_checkpoint(lang, device):\n    files = train_files(lang)\n    ckpt = torch.load(files["model"], map_location=device)\n    cfg = ckpt["config"]\n    model = NNLM(V=cfg["V"], m=cfg["m"], h=cfg["h"], n=cfg["n"], direct=cfg["direct"]).to(device)\n    model.load_state_dict(ckpt["state_dict"])\n    return model, ckpt["stoi"], ckpt["itos"], cfg\n\n\ndef train_language_worker(lang: str, device_id: int, config: dict):\n    set_shared_seed()\n    device = resolve_device(device_id)\n    resources = prepare_language_resources(lang, config)\n\n    model = NNLM(\n        V=len(resources["itos"]),\n        m=config["m"],\n        h=config["h"],\n        n=config["n"],\n        direct=config["direct"],\n    ).to(device)\n\n    history = train_model(\n        model,\n        resources["Xtr"],\n        resources["Ytr"],\n        resources["Xva"],\n        resources["Yva"],\n        device=device,\n        config=config,\n        verbose=True,\n        checkpoint_lang=lang,\n        stoi=resources["stoi"],\n        itos=resources["itos"],\n    )\n\n    plot_path = plot_convergence(history, lang=lang, save_path=train_files(lang)["plot"])\n    save_model_checkpoint(lang, model, config, resources["stoi"], resources["itos"])\n\n    summary = {\n        "lang": lang,\n        "language": language_label(lang),\n        "vocab_size": len(resources["itos"]),\n        "unk_rates": resources["unk_rates"],\n        "split_sizes": resources["split_sizes"],\n        "window_counts": resources["window_counts"],\n        "history": history,\n        "probe_words": pick_words(resources["itos"]),\n        "config": {\n            "n": config["n"],\n            "m": config["m"],\n            "h": config["h"],\n            "direct": config["direct"],\n            "epochs": config["epochs"],\n            "batch_size": config["batch_size"],\n            "min_freq": config["min_freq"],\n            "device_id": device_id,\n        },\n        "plot_path": plot_path,\n        "top_tokens": resources["counter"].most_common(20),\n    }\n\n    with train_files(lang)["state"].open("wb") as f:\n        pickle.dump(summary, f)\n    save_json(train_files(lang)["summary"], summary)\n    print(f"Training stage complete for {language_label(lang)}")\n\n\ndef detect_gpu_count():\n    return torch.cuda.device_count() if torch.cuda.is_available() else 0\n\n\ndef run_parallel_by_language(worker_fn, config: dict, stage_name: str):\n    set_shared_seed()\n    num_gpus = detect_gpu_count()\n    print(f"{stage_name}: visible GPUs = {num_gpus}")\n    if num_gpus > 1:\n        try:\n            mp.set_start_method("spawn", force=True)\n        except RuntimeError:\n            pass\n\n        processes = []\n        for idx, lang in enumerate(LANGS):\n            process = mp.Process(target=worker_fn, args=(lang, idx, copy.deepcopy(config)))\n            process.start()\n            processes.append(process)\n            print(f"Launched {stage_name} worker for {language_label(lang)} on cuda:{idx}")\n\n        exitcodes = []\n        for process in processes:\n            process.join()\n            exitcodes.append(process.exitcode)\n        if any(code != 0 for code in exitcodes):\n            raise RuntimeError(f"{stage_name} multiprocessing failed with exit codes {exitcodes}")\n    else:\n        device_id = 0 if num_gpus == 1 else -1\n        for lang in LANGS:\n            print(f"Running {stage_name} for {language_label(lang)} on device_id={device_id}")\n            worker_fn(lang, device_id, copy.deepcopy(config))\n\n\ndef bundle_files(bundle_name: str, members: list[str | Path]):\n    with zipfile.ZipFile(bundle_name, "w", zipfile.ZIP_DEFLATED) as zf:\n        for member in members:\n            path = Path(member)\n            if path.exists():\n                zf.write(path, arcname=path.name)\n    print(f"Created {bundle_name}")\n\n\ndef stage1_train_all(config: dict):\n    prime_hf_token(config)\n    run_parallel_by_language(train_language_worker, config, "stage1_train")\n\n    members = [EN_CACHE, HI_CACHE]\n    for lang in LANGS:\n        members.extend(train_files(lang).values())\n    bundle_files(STAGE1_BUNDLE, members)\n    print("Stage 1 outputs:")\n    for member in members + [STAGE1_BUNDLE]:\n        if Path(member).exists():\n            print(" -", Path(member).name)\n\n\ndef ensure_stage1_ready(config: dict):\n    if not all(train_files(lang)["model"].exists() for lang in LANGS):\n        extract_bundle_if_present(STAGE1_BUNDLE)\n    missing = [lang for lang in LANGS if not train_files(lang)["model"].exists()]\n    if missing:\n        raise FileNotFoundError(\n            "Missing stage 1 model files for: " + ", ".join(language_label(lang) for lang in missing)\n        )\n\n\ndef metrics_language_worker(lang: str, device_id: int, config: dict):\n    set_shared_seed()\n    device = resolve_device(device_id)\n    resources = prepare_language_resources(lang, config)\n    model, stoi, itos, model_cfg = load_model_checkpoint(lang, device)\n    assert stoi == resources["stoi"], f"Vocabulary mismatch for {lang}; stage1 artifacts do not match current split."\n    assert itos == resources["itos"], f"Vocabulary mismatch for {lang}; stage1 artifacts do not match current split."\n\n    test_ppl = perplexity(model, resources["Xte"], resources["Yte"], device=device, batch_size=config["batch_size"])\n    train_padded = padded_token_sentences(resources["train_sentences"], stoi, n=config["n"])\n    test_padded = padded_token_sentences(resources["test_sentences"], stoi, n=config["n"])\n    ngram_model = Q2_NGramLM(n=config["n"], V=len(itos)).fit(train_padded)\n    laplace_ppl = ngram_model.perplexity(test_padded, mode="laplace")\n    interp_ppl = ngram_model.perplexity(test_padded, mode="interp")\n\n    nnlm_probs = nnlm_token_probs(\n        model,\n        resources["Xte"],\n        resources["Yte"],\n        device=device,\n        batch_size=config["batch_size"],\n    )\n    interp_probs = ngram_model.token_probs(test_padded, mode="interp")\n    mixture = mixture_report_from_probs(nnlm_probs, interp_probs, lam=0.5)\n\n    perplexity_table = pd.DataFrame(\n        [\n            {"model": "NNLM", "perplexity": test_ppl, "lambda": np.nan},\n            {"model": "Laplace n-gram", "perplexity": laplace_ppl, "lambda": np.nan},\n            {"model": "Interpolated n-gram", "perplexity": interp_ppl, "lambda": np.nan},\n            {"model": "Mixture @ 0.50", "perplexity": mixture["mixture_ppl_at_lambda"], "lambda": 0.50},\n            {"model": "Best mixture", "perplexity": mixture["best_mixture_ppl"], "lambda": mixture["best_lambda"]},\n        ]\n    )\n\n    param_counts = count_parameters(\n        V=len(itos),\n        m=model_cfg["m"],\n        h=model_cfg["h"],\n        n=model_cfg["n"],\n        direct=model_cfg["direct"],\n    )\n\n    parameter_table = pd.DataFrame(\n        [\n            {\n                "lang": lang,\n                "language": language_label(lang),\n                "vocab_size": len(itos),\n                "n": model_cfg["n"],\n                "m": model_cfg["m"],\n                "h": model_cfg["h"],\n                "direct": model_cfg["direct"],\n                **param_counts,\n                "ngram_entries_total": ngram_model.distinct_ngrams_total(),\n                **{f"distinct_{order}gram": len(ngram_model.ngram_counts[order]) for order in range(1, config["n"] + 1)},\n            }\n        ]\n    )\n\n    sparsity = sparsity_table(resources["Xtr"], resources["Ytr"], resources["Xte"], resources["Yte"])\n    sparsity["lang"] = lang\n    sparsity["language"] = language_label(lang)\n    sparsity = sparsity[\n        [\n            "lang",\n            "language",\n            "n",\n            "train_contexts_seen",\n            "train_ngrams_seen",\n            "test_tokens",\n            "pct_unseen_context",\n            "pct_unseen_ngram",\n        ]\n    ]\n\n    files = metrics_files(lang)\n    perplexity_table.to_csv(files["perplexity"], index=False)\n    parameter_table.to_csv(files["parameter"], index=False)\n    sparsity.to_csv(files["sparsity"], index=False)\n    mixture["grid"].to_csv(files["mixture"], index=False)\n\n    summary = {\n        "lang": lang,\n        "language": language_label(lang),\n        "perplexities": {\n            "nnlm": test_ppl,\n            "laplace_ngram": laplace_ppl,\n            "interp_ngram": interp_ppl,\n            "mixture_0.5": mixture["mixture_ppl_at_lambda"],\n            "best_mixture": mixture["best_mixture_ppl"],\n            "best_lambda": mixture["best_lambda"],\n        },\n        "parameter_total": int(parameter_table.iloc[0]["TOTAL"]),\n        "ngram_entries_total": int(parameter_table.iloc[0]["ngram_entries_total"]),\n        "vocab_size": int(len(itos)),\n        "sparsity_rows": sparsity.to_dict(orient="records"),\n    }\n    save_json(files["summary"], summary)\n    print(f"Metrics stage complete for {language_label(lang)}")\n\n\ndef stage2_metrics_all(config: dict):\n    ensure_stage1_ready(config)\n    run_parallel_by_language(metrics_language_worker, config, "stage2_metrics")\n\n    members = []\n    for lang in LANGS:\n        members.extend(metrics_files(lang).values())\n    bundle_files(STAGE2_BUNDLE, members)\n    print("Stage 2 outputs:")\n    for member in members + [STAGE2_BUNDLE]:\n        if Path(member).exists():\n            print(" -", Path(member).name)\n\n\ndef ensure_stage2_ready(config: dict):\n    needed = [metrics_files(lang)["summary"] for lang in LANGS]\n    if not all(path.exists() for path in needed):\n        extract_bundle_if_present(STAGE2_BUNDLE)\n    if not all(path.exists() for path in needed):\n        print("Stage 2 outputs missing, computing them now so stage 3 can finish end-to-end.")\n        stage2_metrics_all(config)\n\n\ndef discover_word2vec_artifacts(lang):\n    found = {}\n    for method in ("softmax", "ns", "hs"):\n        path = Path(f"{lang}_embeddings_{method}.pkl")\n        if path.exists():\n            found[method] = load_word2vec(path)\n    return found\n\n\ndef compare_spaces(lang, embeddings, stoi, itos, artifact, method, probe_words):\n    probe_words = [word for word in probe_words if word in stoi and word in artifact["stoi"]]\n    rows = []\n    overlaps = []\n    for word in probe_words:\n        nnlm_neighbors = nearest_neighbours(word, embeddings, stoi, itos, k=10)\n        w2v_neighbors = nearest_neighbours(word, artifact["embeddings"], artifact["stoi"], artifact["itos"], k=10)\n        nnlm_tokens = [tok for tok, _ in nnlm_neighbors]\n        w2v_tokens = [tok for tok, _ in w2v_neighbors]\n        overlap = len(set(nnlm_tokens) & set(w2v_tokens))\n        overlaps.append(overlap / 10.0 if (nnlm_tokens or w2v_tokens) else 0.0)\n        rows.append(\n            {\n                "probe_word": word,\n                "nnlm_neighbors": ", ".join(nnlm_tokens),\n                "word2vec_method": method,\n                "word2vec_neighbors": ", ".join(w2v_tokens),\n                "overlap_at_10": overlap,\n            }\n        )\n\n    compare_df = pd.DataFrame(rows)\n    compare_df.to_csv(analysis_files(lang)["word2vec_compare"], index=False)\n\n    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))\n    for ax, title, E, local_stoi, words in [\n        (axes[0], f"{lang.upper()} NNLM PCA", embeddings, stoi, probe_words),\n        (axes[1], f"{lang.upper()} word2vec ({method}) PCA", artifact["embeddings"], artifact["stoi"], probe_words),\n    ]:\n        labels = [word for word in words if word in local_stoi]\n        if len(labels) >= 2:\n            vectors = np.stack([E[local_stoi[word]] for word in labels], axis=0)\n            centered = vectors - vectors.mean(axis=0, keepdims=True)\n            _, _, vt = np.linalg.svd(centered, full_matrices=False)\n            coords = centered @ vt[:2].T\n            ax.scatter(coords[:, 0], coords[:, 1], s=28, alpha=0.8)\n            for idx, word in enumerate(labels):\n                ax.text(coords[idx, 0], coords[idx, 1], word, fontsize=8)\n        ax.set_title(title)\n        ax.set_xlabel("PC1")\n        ax.set_ylabel("PC2")\n    save_figure(fig, analysis_files(lang)["embed_compare_plot"])\n\n    return {\n        "method": method,\n        "avg_same_word_similarity": same_word_similarity(\n            embeddings, stoi, artifact["embeddings"], artifact["stoi"], probe_words\n        ),\n        "avg_neighbor_overlap": float(np.mean(overlaps)) if overlaps else None,\n        "probe_words": probe_words,\n        "compare_csv": str(analysis_files(lang)["word2vec_compare"]),\n        "compare_plot": str(analysis_files(lang)["embed_compare_plot"]),\n    }\n\n\ndef choose_word2vec_baseline(lang, config, analogies, embeddings, stoi, itos):\n    preferred = config.get("preferred_q3_method", {}).get(lang)\n    artifacts = discover_word2vec_artifacts(lang)\n    if not artifacts:\n        return None, None, "No Q3 embedding artifacts found in the working directory."\n\n    if preferred and preferred in artifacts:\n        artifact = artifacts[preferred]\n        method = preferred\n        note = f"Using user-selected Q3 baseline: {preferred}."\n    else:\n        best = None\n        scored = []\n        for method, artifact in artifacts.items():\n            norm = normalize_rows(artifact["embeddings"])\n            preds = [\n                embedding_analogy_predict(\n                    artifact["embeddings"], norm, artifact["stoi"], artifact["itos"], row.word1, row.word2, row.word3\n                )\n                for row in analogies.itertuples(index=False)\n            ]\n            answered = int(sum(pred != "OOV" for pred in preds))\n            scored.append((answered, {"softmax": 1, "ns": 2, "hs": 3}[method], method))\n        _, _, method = max(scored)\n        artifact = artifacts[method]\n        note = (\n            f"No explicit Q3 baseline was provided, so \'{method}\' was selected as the proxy best model "\n            "because it answered the most analogy rows."\n        )\n\n    artifact_norm = normalize_rows(artifact["embeddings"])\n    proxy_df = analogies.copy()\n    proxy_df["word2vec"] = [\n        embedding_analogy_predict(\n            artifact["embeddings"], artifact_norm, artifact["stoi"], artifact["itos"], row.word1, row.word2, row.word3\n        )\n        for row in analogies.itertuples(index=False)\n    ]\n    proxy_df.to_csv(analysis_files(lang)["word2vec_proxy"], index=False)\n\n    comparison = compare_spaces(lang, embeddings, stoi, itos, artifact, method, pick_words(itos))\n    return comparison, proxy_df, note\n\n\ndef analysis_language_worker(lang: str, device_id: int, config: dict):\n    set_shared_seed()\n    device = resolve_device(device_id)\n    resources = prepare_language_resources(lang, config)\n    model, stoi, itos, _ = load_model_checkpoint(lang, device)\n    assert stoi == resources["stoi"], f"Vocabulary mismatch for {lang}; stage1 artifacts do not match current split."\n    embeddings = model.embeddings()\n\n    probe_words = pick_words(itos)\n    pca_path = plot_pca(\n        embeddings,\n        stoi,\n        probe_words,\n        title=f"{language_label(lang)} NNLM embeddings (PCA)",\n        save_path=analysis_files(lang)["pca"],\n    )\n\n    analogies = load_analogy_set(lang, config)\n    norm = normalize_rows(embeddings)\n    analogy_df = analogies.copy()\n    analogy_df["cbow"] = [\n        embedding_analogy_predict(embeddings, norm, stoi, itos, row.word1, row.word2, row.word3)\n        for row in analogies.itertuples(index=False)\n    ]\n    analogy_df.to_csv(analysis_files(lang)["analogy"], index=False)\n\n    comparison, proxy_df, note = choose_word2vec_baseline(lang, config, analogies, embeddings, stoi, itos)\n    both_answered = 0\n    agreement_rate = None\n    if proxy_df is not None:\n        merged = analogy_df.merge(proxy_df, on=["word1", "word2", "word3"], how="inner")\n        joint = merged[(merged["cbow"] != "OOV") & (merged["word2vec"] != "OOV")].copy()\n        both_answered = int(len(joint))\n        if both_answered:\n            agreement_rate = float((joint["cbow"] == joint["word2vec"]).mean())\n\n    summary = {\n        "lang": lang,\n        "language": language_label(lang),\n        "analogy_answered": int((analogy_df["cbow"] != "OOV").sum()),\n        "analogy_oov": int((analogy_df["cbow"] == "OOV").sum()),\n        "analogy_rows": int(len(analogy_df)),\n        "probe_words": probe_words,\n        "pca_path": pca_path,\n        "word2vec_note": note,\n        "word2vec_comparison": comparison,\n        "word2vec_agreement_rows": both_answered,\n        "word2vec_agreement_rate": agreement_rate,\n    }\n    save_json(analysis_files(lang)["summary"], summary)\n    print(f"Analysis stage complete for {language_label(lang)}")\n\n\ndef build_language_bundle(lang: str):\n    members = [\n        EN_CACHE if lang == "en" else HI_CACHE,\n        train_files(lang)["model"],\n        train_files(lang)["checkpoint"],\n        train_files(lang)["state"],\n        train_files(lang)["summary"],\n        train_files(lang)["plot"],\n        metrics_files(lang)["perplexity"],\n        metrics_files(lang)["parameter"],\n        metrics_files(lang)["sparsity"],\n        metrics_files(lang)["mixture"],\n        metrics_files(lang)["summary"],\n        analysis_files(lang)["analogy"],\n        analysis_files(lang)["pca"],\n        analysis_files(lang)["word2vec_compare"],\n        analysis_files(lang)["word2vec_proxy"],\n        analysis_files(lang)["embed_compare_plot"],\n        analysis_files(lang)["written_answers"],\n        analysis_files(lang)["summary"],\n    ]\n    bundle_files(train_files(lang)["bundle"], members)\n    return train_files(lang)["bundle"]\n\n\ndef build_language_answer_draft(lang: str):\n    train_state = load_train_state(lang)\n    metric = load_json(metrics_files(lang)["summary"])\n    analysis = load_json(analysis_files(lang)["summary"])\n\n    hist = train_state["history"]\n    if hist["converged"]:\n        convergence_note = (\n            f"The model early-stopped at epoch {hist[\'stopped_epoch\']} and reached its best validation "\n            f"perplexity at epoch {hist[\'best_epoch\']}."\n        )\n    else:\n        convergence_note = (\n            f"The run hit the epoch cap at {hist[\'stopped_epoch\']}; check the convergence plot to decide "\n            "whether more epochs are still needed."\n        )\n\n    text = f"""# Question 4 {language_label(lang)} draft\n\n## Step 7\n\n1. Mixture perplexity at lambda = 0.5: {metric[\'perplexities\'][\'mixture_0.5\']:.4f}\n2. Best mixture perplexity: {metric[\'perplexities\'][\'best_mixture\']:.4f} at lambda = {metric[\'perplexities\'][\'best_lambda\']:.2f}\n3. The mixture {"does" if metric[\'perplexities\'][\'best_mixture\'] < min(metric[\'perplexities\'][\'nnlm\'], metric[\'perplexities\'][\'interp_ngram\']) else "does not"} beat both standalone components for {language_label(lang)}.\n\n## Observations\n\n1. NNLM perplexity = {metric[\'perplexities\'][\'nnlm\']:.4f}, Laplace n-gram = {metric[\'perplexities\'][\'laplace_ngram\']:.4f}, interpolated n-gram = {metric[\'perplexities\'][\'interp_ngram\']:.4f}.\n2. {convergence_note}\n3. NNLM parameters = {metric[\'parameter_total\']:,}; stored n-gram entries = {metric[\'ngram_entries_total\']:,}.\n4. Unseen-context percentages are listed in `{metrics_files(lang)[\'sparsity\'].name}`.\n5. Cross-language hardness still needs both English and Hindi notebooks together before writing the final combined answer.\n6. {analysis[\'word2vec_note\']}\n\nAdditional notes:\n- Analogy rows answered: {analysis[\'analogy_answered\']} / {analysis[\'analogy_rows\']}\n- Analogy OOV rows: {analysis[\'analogy_oov\']}\n- Agreement rows with selected Q3 word2vec model: {analysis[\'word2vec_agreement_rows\']}\n- Agreement rate with selected Q3 word2vec model: {fmt_optional(analysis[\'word2vec_agreement_rate\'])}\n"""\n    analysis_files(lang)["written_answers"].write_text(text, encoding="utf-8")\n    return analysis_files(lang)["written_answers"]\n\n\ndef run_full_language_pipeline(lang: str, config: dict, device_id: int | None = None):\n    expected_inputs = [\n        EN_CACHE,\n        HI_CACHE,\n        train_files(lang)["checkpoint"].name,\n        train_files(lang)["model"].name,\n    ]\n    for method in ("softmax", "ns", "hs"):\n        expected_inputs.append(f"{lang}_embeddings_{method}.pkl")\n\n    stage_kaggle_input_files(expected_inputs)\n    prime_hf_token(config)\n    if device_id is None:\n        device_id = 0 if detect_gpu_count() > 0 else -1\n\n    print(f"Running full Q4 pipeline for {language_label(lang)} on device_id={device_id}")\n    train_language_worker(lang, device_id, copy.deepcopy(config))\n    metrics_language_worker(lang, device_id, copy.deepcopy(config))\n    analysis_language_worker(lang, device_id, copy.deepcopy(config))\n    build_language_answer_draft(lang)\n    bundle = build_language_bundle(lang)\n\n    print(f"Completed full Q4 pipeline for {language_label(lang)}")\n    print("Artifacts:")\n    for path in [\n        train_files(lang)["model"],\n        train_files(lang)["checkpoint"],\n        train_files(lang)["plot"],\n        metrics_files(lang)["perplexity"],\n        metrics_files(lang)["parameter"],\n        metrics_files(lang)["sparsity"],\n        metrics_files(lang)["mixture"],\n        analysis_files(lang)["analogy"],\n        analysis_files(lang)["pca"],\n        analysis_files(lang)["word2vec_compare"],\n        analysis_files(lang)["word2vec_proxy"],\n        analysis_files(lang)["embed_compare_plot"],\n        analysis_files(lang)["written_answers"],\n        bundle,\n    ]:\n        if Path(path).exists():\n            print(" -", Path(path).name)\n\n\ndef fmt_optional(value):\n    return "nan" if value is None else f"{value:.4f}"\n\n\ndef load_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef load_train_state(lang):\n    with train_files(lang)["state"].open("rb") as f:\n        return pickle.load(f)\n\n\ndef draft_answers(train_states, metric_summaries, analysis_summaries):\n    en_train = train_states["en"]\n    hi_train = train_states["hi"]\n    en_metric = metric_summaries["en"]\n    hi_metric = metric_summaries["hi"]\n    en_analysis = analysis_summaries["en"]\n    hi_analysis = analysis_summaries["hi"]\n\n    def conv_text(state):\n        hist = state["history"]\n        if hist["converged"]:\n            return (\n                f"Yes. Early stopping triggered after epoch {hist[\'stopped_epoch\']}, "\n                f"and the best validation perplexity occurred at epoch {hist[\'best_epoch\']}."\n            )\n        return (\n            f"Not fully guaranteed. The run hit the epoch cap at {hist[\'stopped_epoch\']}, "\n            "so the reported perplexity should be treated as a lower bound unless the validation curve is flat."\n        )\n\n    harder = "Hindi" if hi_metric["perplexities"]["nnlm"] > en_metric["perplexities"]["nnlm"] else "English"\n    easier = "English" if harder == "Hindi" else "Hindi"\n    harder_train = hi_train if harder == "Hindi" else en_train\n    easier_train = en_train if harder == "Hindi" else hi_train\n    harder_metric = hi_metric if harder == "Hindi" else en_metric\n    easier_metric = en_metric if harder == "Hindi" else hi_metric\n\n    return f"""# Question 4 written-answer draft\n\n## Step 7\n\n### 1. Mixture perplexity at $\\\\lambda = 0.5$\n- English: the NNLM/interpolated n-gram mixture reached perplexity {en_metric[\'perplexities\'][\'mixture_0.5\']:.4f} at $\\\\lambda = 0.5$.\n- Hindi: the NNLM/interpolated n-gram mixture reached perplexity {hi_metric[\'perplexities\'][\'mixture_0.5\']:.4f} at $\\\\lambda = 0.5$.\n\n### 2. Best mixture after sweeping $\\\\lambda$\n- English: the best mixture perplexity was {en_metric[\'perplexities\'][\'best_mixture\']:.4f} at $\\\\lambda = {en_metric[\'perplexities\'][\'best_lambda\']:.2f}$.\n- Hindi: the best mixture perplexity was {hi_metric[\'perplexities\'][\'best_mixture\']:.4f} at $\\\\lambda = {hi_metric[\'perplexities\'][\'best_lambda\']:.2f}$.\n\n### 3. Does the mixture beat both components?\n- English: {"yes" if en_metric[\'perplexities\'][\'best_mixture\'] < min(en_metric[\'perplexities\'][\'nnlm\'], en_metric[\'perplexities\'][\'interp_ngram\']) else "no"}. This suggests the NNLM and the n-gram model make partly different mistakes, so interpolation benefits from their complementarity.\n- Hindi: {"yes" if hi_metric[\'perplexities\'][\'best_mixture\'] < min(hi_metric[\'perplexities\'][\'nnlm\'], hi_metric[\'perplexities\'][\'interp_ngram\']) else "no"}. A weaker standalone model can still improve the mixture if it is strong on tokens where the other model fails.\n\n## Observations\n\n### Question 1\n- English: NNLM perplexity = {en_metric[\'perplexities\'][\'nnlm\']:.4f}, Laplace n-gram = {en_metric[\'perplexities\'][\'laplace_ngram\']:.4f}, interpolated n-gram = {en_metric[\'perplexities\'][\'interp_ngram\']:.4f}.\n- Hindi: NNLM perplexity = {hi_metric[\'perplexities\'][\'nnlm\']:.4f}, Laplace n-gram = {hi_metric[\'perplexities\'][\'laplace_ngram\']:.4f}, interpolated n-gram = {hi_metric[\'perplexities\'][\'interp_ngram\']:.4f}.\n\n### Question 2\n- English: {conv_text(en_train)}\n- Hindi: {conv_text(hi_train)}\n\n### Question 3\n- English: the NNLM used {en_metric[\'parameter_total\']:,} learned parameters, while the count model stored {en_metric[\'ngram_entries_total\']:,} distinct n-gram entries across orders.\n- Hindi: the NNLM used {hi_metric[\'parameter_total\']:,} learned parameters, while the count model stored {hi_metric[\'ngram_entries_total\']:,} distinct n-gram entries across orders.\n\n### Question 4\n- English unseen-context / unseen-ngram percentages by order:\n{pd.DataFrame(en_metric[\'sparsity_rows\'])[[\'n\', \'pct_unseen_context\', \'pct_unseen_ngram\']].to_string(index=False)}\n\n- Hindi unseen-context / unseen-ngram percentages by order:\n{pd.DataFrame(hi_metric[\'sparsity_rows\'])[[\'n\', \'pct_unseen_context\', \'pct_unseen_ngram\']].to_string(index=False)}\n\nThe NNLM can still assign sensible probability to unseen contexts because it uses shared word embeddings and shared network parameters instead of storing one isolated probability table entry per context.\n\n### Question 5\n- Within this shared preprocessing pipeline, {harder} was harder to model because its NNLM perplexity was higher ({harder_metric[\'perplexities\'][\'nnlm\']:.4f} vs {easier_metric[\'perplexities\'][\'nnlm\']:.4f} for {easier}). Part of that difference is explained by vocabulary size ({harder_metric[\'vocab_size\']:,} vs {easier_metric[\'vocab_size\']:,}) and test `<unk>` rate ({harder_train[\'unk_rates\'][\'test\']:.4%} vs {easier_train[\'unk_rates\'][\'test\']:.4%}).\n\n### Question 6\n- English word2vec baseline note: {en_analysis[\'word2vec_note\']}\n- Hindi word2vec baseline note: {hi_analysis[\'word2vec_note\']}\n- English same-word cosine similarity vs selected word2vec space: {fmt_optional(None if en_analysis[\'word2vec_comparison\'] is None else en_analysis[\'word2vec_comparison\'][\'avg_same_word_similarity\'])}.\n- Hindi same-word cosine similarity vs selected word2vec space: {fmt_optional(None if hi_analysis[\'word2vec_comparison\'] is None else hi_analysis[\'word2vec_comparison\'][\'avg_same_word_similarity\'])}.\n- English average top-10 neighbour overlap with selected word2vec space: {fmt_optional(None if en_analysis[\'word2vec_comparison\'] is None else en_analysis[\'word2vec_comparison\'][\'avg_neighbor_overlap\'])}.\n- Hindi average top-10 neighbour overlap with selected word2vec space: {fmt_optional(None if hi_analysis[\'word2vec_comparison\'] is None else hi_analysis[\'word2vec_comparison\'][\'avg_neighbor_overlap\'])}.\n\nThe NNLM and word2vec spaces usually agree on broad semantic neighborhoods, but they differ in exact neighbour rankings because the objectives are different: word2vec optimizes local embedding relations directly, while the NNLM learns embeddings only as part of next-word prediction.\n"""\n\n\ndef combine_final_outputs(config: dict):\n    train_states = {lang: load_train_state(lang) for lang in LANGS}\n    metric_summaries = {lang: load_json(metrics_files(lang)["summary"]) for lang in LANGS}\n    analysis_summaries = {lang: load_json(analysis_files(lang)["summary"]) for lang in LANGS}\n\n    parameter_table = pd.concat(\n        [pd.read_csv(metrics_files(lang)["parameter"]) for lang in LANGS],\n        ignore_index=True,\n    )\n    parameter_table.to_csv("q4_parameter_count_table.csv", index=False)\n\n    sparsity = pd.concat(\n        [pd.read_csv(metrics_files(lang)["sparsity"]) for lang in LANGS],\n        ignore_index=True,\n    )\n    sparsity.to_csv("q4_sparsity_table.csv", index=False)\n\n    summary_rows = []\n    for lang in LANGS:\n        t = train_states[lang]\n        m = metric_summaries[lang]\n        a = analysis_summaries[lang]\n        summary_rows.append(\n            {\n                "lang": lang,\n                "language": language_label(lang),\n                "vocab_size": m["vocab_size"],\n                "unk_rate_train": t["unk_rates"]["train"],\n                "unk_rate_valid": t["unk_rates"]["valid"],\n                "unk_rate_test": t["unk_rates"]["test"],\n                "nnlm_ppl": m["perplexities"]["nnlm"],\n                "laplace_ppl": m["perplexities"]["laplace_ngram"],\n                "interp_ppl": m["perplexities"]["interp_ngram"],\n                "mixture_0_5_ppl": m["perplexities"]["mixture_0.5"],\n                "best_mixture_ppl": m["perplexities"]["best_mixture"],\n                "best_lambda": m["perplexities"]["best_lambda"],\n                "analogy_answered": a["analogy_answered"],\n                "analogy_oov": a["analogy_oov"],\n                "word2vec_agreement_rows": a["word2vec_agreement_rows"],\n                "word2vec_agreement_rate": a["word2vec_agreement_rate"],\n            }\n        )\n\n    pd.DataFrame(summary_rows).to_csv("q4_language_summary.csv", index=False)\n    draft = draft_answers(train_states, metric_summaries, analysis_summaries)\n    Path("Q4_WRITTEN_ANSWERS.md").write_text(draft, encoding="utf-8")\n    print(draft)\n\n    final_members = [\n        "analogy_results_nnlm_en.csv",\n        "analogy_results_nnlm_hi.csv",\n        "q4_convergence_en.png",\n        "q4_convergence_hi.png",\n        "q4_perplexity_table_en.csv",\n        "q4_perplexity_table_hi.csv",\n        "q4_parameter_table_en.csv",\n        "q4_parameter_table_hi.csv",\n        "q4_sparsity_table_en.csv",\n        "q4_sparsity_table_hi.csv",\n        "q4_mixture_grid_en.csv",\n        "q4_mixture_grid_hi.csv",\n        "q4_parameter_count_table.csv",\n        "q4_sparsity_table.csv",\n        "q4_embedding_pca_en.png",\n        "q4_embedding_pca_hi.png",\n        "q4_word2vec_compare_en.csv",\n        "q4_word2vec_compare_hi.csv",\n        "q4_word2vec_proxy_en.csv",\n        "q4_word2vec_proxy_hi.csv",\n        "q4_embedding_compare_en.png",\n        "q4_embedding_compare_hi.png",\n        "q4_language_summary.csv",\n        "Q4_WRITTEN_ANSWERS.md",\n        "nnlm_en.pt",\n        "nnlm_hi.pt",\n    ]\n    bundle_files(FINAL_BUNDLE, final_members)\n    print("Final Question 4 bundle contents:")\n    for member in final_members:\n        if Path(member).exists():\n            print(" -", member)\n\n\ndef stage3_analysis_and_bundle(config: dict):\n    ensure_stage1_ready(config)\n    ensure_stage2_ready(config)\n    run_parallel_by_language(analysis_language_worker, config, "stage3_analysis")\n    combine_final_outputs(config)\n', encoding="utf-8")
print("Prepared q4_parallel_runtime.py in the current session.")


## Runtime Config

This notebook is pinned to `cuda:0` when that device is visible.

For the Question 4 word2vec-comparison deliverables, also upload the relevant
Q3 embedding pickle files for this language into the same Kaggle session
through Kaggle's Input pane:
- English notebook: `en_embeddings_softmax.pkl`, `en_embeddings_ns.pkl`, `en_embeddings_hs.pkl`
- Hindi notebook: `hi_embeddings_softmax.pkl`, `hi_embeddings_ns.pkl`, `hi_embeddings_hs.pkl`


In [ ]:
import os
from q4_parallel_runtime import build_default_config

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
    except Exception:
        HF_TOKEN = ""

CONFIG = build_default_config()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    CONFIG["hf_token"] = HF_TOKEN

CONFIG["max_pairs"] = None
CONFIG["n"] = 4
CONFIG["m"] = 100
CONFIG["h"] = 100
CONFIG["epochs"] = 30
CONFIG["batch_size"] = 512
CONFIG["lr"] = 1e-3
CONFIG["weight_decay"] = 1e-4
CONFIG["patience"] = 3
CONFIG["min_freq"] = 2
CONFIG["direct"] = False
CONFIG["preferred_q3_method"] = {"en": None, "hi": None}

LANG = "en"
DEVICE_ID = 0

CONFIG


## Run Full English Pipeline

This produces all per-language Q4 artifacts and the download zip.


In [ ]:
from q4_parallel_runtime import run_full_language_pipeline

run_full_language_pipeline(LANG, CONFIG, device_id=DEVICE_ID)


In [ ]:
from pathlib import Path

path = Path("q4_parallel_runtime.py")
text = path.read_text(encoding="utf-8")
text = text.replace(
    "optimizer_state = optimizer.state_dict()",
    "optimizer_state = copy.deepcopy(optimizer.state_dict())"
)
path.write_text(text, encoding="utf-8")
print("Patched q4_parallel_runtime.py")

In [ ]:
import importlib
import q4_parallel_runtime

importlib.reload(q4_parallel_runtime)
from q4_parallel_runtime import run_full_language_pipeline

In [ ]:
run_full_language_pipeline(LANG, CONFIG, device_id=DEVICE_ID)